In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # tested first, then switched to RandomForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from mlflow.models.signature import infer_signature
import matplotlib.pyplot as plt


# Load data from gold layer and convert to Pandas
df = spark.table("gold.pcos_features").toPandas()

df['is_menstrual_irregular'] = df['is_menstrual_irregular'].astype(int)
df['has_pcos'] = df['has_pcos'].astype(int)

# Define features and target
X = (df.drop(columns=["has_pcos"])).astype(float)
y = df["has_pcos"]


# Split Data into Train and Test Sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=49
)

# Start an MLflow Run (Built-in Databricks Tracking)
with mlflow.start_run(run_name="PCOS_Random_Forest"):
    
    # Train Random Forest Classifier
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=51)
    model.fit(X_train, y_train)

    # Infer the signature (This captures the input/output schema)
    signature = infer_signature(X_train, model.predict(X_train))
    
    # Define an input example for model deployment
    input_example = X_train.head(3)
    
    # Evaluate
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred)
    }
    
    # Log Metrics and Model to Databricks
    mlflow.log_params({"max_iter": 1000, "random_state": 2})
    mlflow.log_metrics(metrics)
  
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="pcos_model",
        signature=signature,
        input_example=input_example
    )
    
    print(f"Model trained and logged to MLflow with accuracy: {metrics['accuracy']}")

Model trained and logged to MLflow with accuracy: 1.0


The accuracy is clearly way too high. The dataset I'm using is too perfect, so the model has an easy time adapting. chanigng the depth and complexity of the model, reduces the accuracy.

In [0]:


# Used this to visualize feature importance, to figure out which features are most relevant
importances = model.feature_importances_
feature_names = ["age", "bmi", "is_menstrual_irregular", "testosterone_level", "follicle_count"]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False))

                  Feature  Importance
2  is_menstrual_irregular    0.329870
1                     bmi    0.296664
3      testosterone_level    0.202404
4          follicle_count    0.155254
0                     age    0.015808
